<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
DeepAgents — Multi-Skill Progressive Disclosure
</b></font> </br></p>

---

In [28]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import json
import os
import urllib.request
from pathlib import Path

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M36-Multi-Skill"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import check_environment, setup_api_keys, mprint, mermaid

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"])
check_environment()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-anthropic                      1.4.0
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-google-genai                   4.2.1
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11


In [29]:
#@title 📦 Installationen { display-mode: "form" }

# Getestete Versionen (Stand: März 2026)
# deepagents 0.4.x: skills=[...]-Parameter ist stabil ab v0.4.10
# ⚠️ Vor einem Update: CHANGELOG prüfen — Sub-Agent-Dict-Syntax kann sich ändern
!uv pip install --system -q "deepagents==0.4.12"

# 1 | Ziel und Lernidee

---

<font color='black' size='5'>Von einem Skill zu vielen</font>

In **M33** haben wir einen einzelnen Skill — Meeting-Briefing — manuell aus GitHub geladen,
in einen System-Prompt injiziert und über ein Gate-Writer-Pattern gesteuert.
Volle Kontrolle, aber alles explizit verdrahtet.

**M36 zeigt den komplementären Ansatz:** DeepAgents scannt ein lokales Skills-Verzeichnis
beim Start, liest die `description`-Felder aus allen `SKILL.md`-Dateien und entscheidet
bei jeder Anfrage dynamisch, welcher Skill passt — **Progressive Disclosure**.

| Lernziel | Beschreibung |
|----------|--------------|
| Skills-Scan | Agent liest SKILL.md-Frontmatter automatisch beim Start |
| Skill-Routing | Dynamische Auswahl anhand von Trigger-Phrasen |
| GitHub-Cache | Skill-Dateien aus GitHub laden, lokal cachen, versionieren |
| Abgrenzung | Vergleich mit M33 (manuell) vs. M36 (nativ) |

> 📌 **Dieses Notebook lehrt Skill-Routing — nicht Gate-Writer-Architektur.**
> Für Gate-Writer → M33.

In [30]:
#@markdown   <p><font size="4" color="green">Architektur-Übersicht</font></p>

diagram_arch = '''
%%{init: {'theme':'light'}}%%
flowchart TB
    GH([" 🐙 GitHub Repo\nralf-42/Agenten"]) -->|"wget + cache"| CACHE

    subgraph CACHE [" 📁 Lokaler Skills-Cache"]
        S1["compliance/SKILL.md"]
        S2["meeting-briefing/SKILL.md"]
        S3["research/SKILL.md"]
    end

    CACHE -->|"skills=[cache_dir]"| AGENT

    subgraph AGENT [" 🤖 DeepAgent Koordinator — o3"]
        SCAN["Skill-Scan\n(Frontmatter lesen)"] --> INJECT["descriptions →\nSystem-Prompt"]
        INJECT --> ROUTE["Anfrage analysieren\nSkill auswählen"]
    end

    U([" 🧑 Nutzeranfrage"]) --> ROUTE
    ROUTE -->|"compliance"| R1[" ✅ Compliance-Workflow"]
    ROUTE -->|"meeting"| R2[" 📋 Briefing-Workflow"]
    ROUTE -->|"research"| R3[" 🔍 Recherche-Workflow"]
'''
mermaid(diagram_arch, width=650)

# 2 | Setup

---

In [31]:
from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field

from genai_lib.model_config import ROUTER, WORKER  # o3-mini, gpt-5.4-mini

# Koordinator: o3-mini (Demo-Routing, kein temperature)
# o3 wäre zu scharf für diesen Demo-Kontext und hat engeres TPM-Limit (30k)
coord_llm = init_chat_model(ROUTER)

mprint("✅ Pakete geladen")

✅ Pakete geladen

# 3 | GitHub-Skills laden und lokal cachen

---

Alle `SKILL.md`-Dateien werden aus dem GitHub-Repo `ralf-42/Agenten` geladen
und lokal in `./_skills_cache/` gespeichert.

**Warum lokaler Cache statt direkter GitHub-URL?**

| Aspekt | Direkte URL | Lokaler Cache |
|--------|-------------|---------------|
| Reproduzierbarkeit | ❌ `main` kann sich ändern | ✅ Commit-SHA pinnen |
| Offline-Betrieb | ❌ | ✅ |
| `skills=[...]`-API | ❌ (erwartet lokales FS) | ✅ |
| GitHub-Rate-Limit | ⚠️ bei vielen Runs | ✅ einmal laden |

> 💡 **Commit-SHA pinnen** für vollständige Reproduzierbarkeit:
> `COMMIT_SHA = "abc1234"` statt `"main"`

In [32]:
# 3.1 Skill-Dateien aus GitHub laden und lokal cachen

GITHUB_RAW  = "https://raw.githubusercontent.com/ralf-42/Agenten"
COMMIT_SHA  = "main"  # 💡 Für Reproduzierbarkeit: festes SHA eintragen
SKILL_BASE  = f"{GITHUB_RAW}/{COMMIT_SHA}/06_skill"

CACHE_DIR   = Path("./_skills_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Zu ladende Skills: (Skill-Name, Dateipfad im Repo)
SKILL_FILES = [
    ("compliance",       "compliance/SKILL.md"),
    ("meeting-briefing", "meeting-briefing/SKILL.md"),
    ("research",         "research/SKILL.md"),
]

def download_skill(skill_name: str, rel_path: str) -> Path:
    """Lädt eine SKILL.md-Datei aus GitHub und speichert sie lokal."""
    url        = f"{SKILL_BASE}/{rel_path}"
    local_dir  = CACHE_DIR / skill_name
    local_dir.mkdir(exist_ok=True)
    local_file = local_dir / "SKILL.md"
    with urllib.request.urlopen(url) as resp:
        local_file.write_bytes(resp.read())
    return local_file

# Download ausführen
cached_files = {}
for skill_name, rel_path in SKILL_FILES:
    path = download_skill(skill_name, rel_path)
    cached_files[skill_name] = path
    print(f"  ✅ {skill_name:20s} → {path}")

print(f"\nCache-Verzeichnis: {CACHE_DIR.resolve()}")

  ✅ compliance           → _skills_cache/compliance/SKILL.md
  ✅ meeting-briefing     → _skills_cache/meeting-briefing/SKILL.md
  ✅ research             → _skills_cache/research/SKILL.md

Cache-Verzeichnis: /content/_skills_cache


# 4 | Lokales Skills-Verzeichnis inspizieren

---

Bevor der Agent gebaut wird, zeigen wir explizit, was er beim Start sieht:
die `name`- und `description`-Felder aus dem YAML-Frontmatter jeder `SKILL.md`.

Genau diese `description`-Werte werden in den System-Prompt injiziert,
sodass der Agent bei einer Anfrage erkennt, welcher Skill relevant ist.

In [45]:
# 4.1 Frontmatter aus SKILL.md-Dateien lesen und anzeigen

import re

def read_frontmatter(path: Path) -> dict:
    """Liest YAML-Frontmatter aus einer Markdown-Datei."""
    text = path.read_text(encoding="utf-8")
    match = re.match(r"^---\n(.+?)\n---", text, re.DOTALL)
    if not match:
        return {}
    result = {}
    for line in match.group(1).split("\n"):
        if ":" in line:
            key, _, val = line.partition(":")
            result[key.strip()] = val.strip()
    return result

# Alle gecachten Skills inspizieren
zeilen = [
    "## 📁 Skills-Verzeichnis: was der Agent beim Start sieht",
    "",
    "| Skill | name | description (Auszug) |",
    "|-------|------|----------------------|",
]
for skill_name, path in cached_files.items():
    fm = read_frontmatter(path)
    name = fm.get("name", "—")
    desc = fm.get("description", "—")[:280] + "..."
    zeilen.append(f"| `{skill_name}` | `{name}` | {desc} |")

mprint("\n".join(zeilen))

## 📁 Skills-Verzeichnis: was der Agent beim Start sieht

| Skill | name | description (Auszug) |
|-------|------|----------------------|
| `compliance` | `compliance-skill` | >-... |
| `meeting-briefing` | `meeting-briefing-skill` | >-... |
| `research` | `research-skill` | >-... |

# 5 | DeepAgent mit skills=[...] erzeugen

---

Die `skills=[...]`-API erwartet **lokale Verzeichnispfade**. Der Agent scannt
beim ersten Invoke alle Unterordner nach `SKILL.md`-Dateien und injiziert
deren `description`-Felder in den System-Prompt.

**Wichtig:** `skills_files` übergibt den Datei-Inhalt an das virtuelle Dateisystem
des Agents. Der Pfad-Prefix `/skills/` im Dict muss mit dem `skills=["/skills/"]`-
Parameter übereinstimmen.

```python
# Minimales Muster
agent = create_deep_agent(
    model=llm,
    skills=["/skills/"],   # virtueller FS-Pfad
    checkpointer=MemorySaver(),
)
result = agent.invoke(
    {"messages": [...], "files": skills_files},
    config={"configurable": {"thread_id": "..."}, "recursion_limit": 50},
)
```

In [34]:
# 5.1 Einfache Mock-Tools (Demo-Zwecke: zeigt Skill-Routing, nicht volle Logik)

@tool
def compliance_check(subject_name: str, country: str, transaction_amount: float = 0.0) -> str:
    """
    Berechnet den Compliance-Risiko-Score für ein Subjekt.
    Gibt low, medium oder high zurück.
    """
    # Demo: einfache Heuristik — in Produktion: echte Sanktionsdatenbank
    high_risk_countries = {"IR", "KP", "SY", "RU", "BY"}
    risk = "high" if country.upper() in high_risk_countries else (
        "medium" if transaction_amount > 50_000 else "low"
    )
    return json.dumps({"risk_level": risk, "subject": subject_name, "country": country})


@tool
def extract_actions(text: str) -> str:
    """
    Extrahiert Action Items aus Meeting-Text deterministisch.
    Gibt JSON-Liste zurück.
    """
    # Demo: Schlüsselwörter-Heuristik
    actions = []
    keywords = ["bis", "bis zum", "deadline", "verantwortlich", "action"]
    for line in text.split("\n"):
        if any(kw in line.lower() for kw in keywords):
            actions.append({"task": line.strip()[:80], "owner": "[offen]", "due": "[offen]"})
    if not actions:
        actions = [{"task": "Keine Action Items gefunden", "owner": "—", "due": "—"}]
    return json.dumps(actions, ensure_ascii=False)


@tool
def score_relevance(source_text: str, query: str) -> str:
    """
    Bewertet die Relevanz einer Quelle für eine Suchanfrage.
    Gibt Score zwischen 0.0 und 1.0 zurück.
    """
    # Demo: Überlappung von Wörtern als Relevanz-Proxy
    query_words = set(query.lower().split())
    source_words = set(source_text.lower().split())
    overlap = len(query_words & source_words)
    score = min(1.0, overlap / max(len(query_words), 1) * 1.5)
    return json.dumps({"score": round(score, 2), "query": query})


TOOLS = [compliance_check, extract_actions, score_relevance]
mprint(f"✅ {len(TOOLS)} Mock-Tools registriert")

✅ 3 Mock-Tools registriert

In [35]:
# 5.2 skills_files-Dict aufbauen (virtuelles Dateisystem)

VIRT_PREFIX = "/skills/"  # muss mit skills=[VIRT_PREFIX] übereinstimmen

skills_files = {}
for skill_name, local_path in cached_files.items():
    virt_path = f"{VIRT_PREFIX}{skill_name}/SKILL.md"
    content   = local_path.read_text(encoding="utf-8")
    skills_files[virt_path] = create_file_data(content)
    print(f"  📄 {virt_path}")

mprint(f"\n✅ {len(skills_files)} Skill-Dateien im virtuellen FS registriert")

  📄 /skills/compliance/SKILL.md
  📄 /skills/meeting-briefing/SKILL.md
  📄 /skills/research/SKILL.md



✅ 3 Skill-Dateien im virtuellen FS registriert

In [36]:
# 5.3 DeepAgent mit nativer skills=[...]-API erzeugen

import time

multi_skill_agent = create_deep_agent(
    model=coord_llm,
    tools=TOOLS,
    skills=[VIRT_PREFIX],          # ← native Skills-API
    checkpointer=MemorySaver(),
    system_prompt=(
        "Du bist ein Allround-Assistent mit Zugriff auf spezialisierte Skills. "
        "Wähle bei jeder Anfrage den passenden Skill aus den verfügbaren Skills. "
        "Wenn kein Skill passt, antworte direkt ohne Skill. "
        "Antworte immer auf Deutsch."
    ),
)


def invoke_agent(anfrage: str, session_id: str = None) -> str:
    """Hilfsfunktion: Agent aufrufen und letzte Antwort zurückgeben."""
    sid = session_id or f"m36-{int(time.time())}"
    result = multi_skill_agent.invoke(
        {
            "messages": [{"role": "user", "content": anfrage}],
            "files":    skills_files,
        },
        config={
            "configurable": {"thread_id": sid},
            "recursion_limit": 50,
        },
    )
    return result["messages"][-1].content


mprint("✅ Multi-Skill-Agent bereit")

✅ Multi-Skill-Agent bereit

# 6 | Demo 1 — Compliance-Anfrage

---

**Erwartetes Verhalten:** Der Agent erkennt an der Formulierung
("Lieferantenprüfung", "onboarding", "Risikobewertung"), dass der
`compliance-skill` passt, liest die vollständige `SKILL.md` und
führt den definierten Workflow aus.

> 📌 Progressive Disclosure: Erst die `description` wird gelesen.
> Erst bei einem Match liest der Agent die volle SKILL.md.

In [37]:
# Demo 1: Compliance — Lieferantenprüfung

anfrage_compliance = """
Ich muss einen neuen Lieferanten onboarden: Firma SteelTech GmbH aus Deutschland,
Auftragswert 35.000 EUR, Bereich Maschinenbauteile.
Sanktionsprüfung wurde durchgeführt, kein Treffer.
Bitte führe eine Risikobewertung durch und gib eine Freigabe-Empfehlung.
"""

print("📋 Anfrage:")
print(anfrage_compliance.strip())
print("\n" + "─" * 60 + "\n")

antwort_1 = invoke_agent(anfrage_compliance, session_id="demo1-compliance")

print("🤖 Agent-Antwort:")
mprint(antwort_1)

📋 Anfrage:
Ich muss einen neuen Lieferanten onboarden: Firma SteelTech GmbH aus Deutschland,
Auftragswert 35.000 EUR, Bereich Maschinenbauteile.
Sanktionsprüfung wurde durchgeführt, kein Treffer.
Bitte führe eine Risikobewertung durch und gib eine Freigabe-Empfehlung.

────────────────────────────────────────────────────────────

🤖 Agent-Antwort:


Die Risikobewertung ergibt ein niedriges Risiko ("low"). Die Sanktionsprüfung war negativ, sodass ich unter Berücksichtigung des niedrigen Risikos die Freigabe des Lieferanten SteelTech GmbH empfehle.

# 7 | Demo 2 — Meeting-Briefing-Anfrage

---

**Erwartetes Verhalten:** Trigger-Phrasen wie "Meeting vorbereiten",
"Agenda", "Briefing" matchen den `meeting-briefing-skill`.
Der Skill-Workflow strukturiert die Agenda und extrahiert Action Items.

In [38]:
# Demo 2: Meeting-Briefing — Sprint-Review vorbereiten

anfrage_meeting = """
Bereite das Sprint-Review für morgen vor.
Teilnehmer: Anna (PO), Ben (Dev), Clara (QA), David (Stakeholder).
Themen: Velocity war 42 SP, 3 Stories offen wegen API-Problemen,
Deployment für Freitag geplant.
Offene Action Items aus dem letzten Meeting: Ben liefert API-Fix bis heute,
Clara erstellt Test-Report bis Donnerstag.
"""

print("📋 Anfrage:")
print(anfrage_meeting.strip())
print("\n" + "─" * 60 + "\n")

antwort_2 = invoke_agent(anfrage_meeting, session_id="demo2-meeting")

print("🤖 Agent-Antwort:")
mprint(antwort_2)

📋 Anfrage:
Bereite das Sprint-Review für morgen vor.
Teilnehmer: Anna (PO), Ben (Dev), Clara (QA), David (Stakeholder).
Themen: Velocity war 42 SP, 3 Stories offen wegen API-Problemen,
Deployment für Freitag geplant.
Offene Action Items aus dem letzten Meeting: Ben liefert API-Fix bis heute,
Clara erstellt Test-Report bis Donnerstag.

────────────────────────────────────────────────────────────

🤖 Agent-Antwort:


Hier ist das vorbereitete Briefing für das Sprint-Review:

Teilnehmer:
• Anna (PO)
• Ben (Developer)
• Clara (QA)
• David (Stakeholder)

Agenda:
1. Begrüßung und kurze Einführung.
2. Rückblick auf den Sprint:
   • Velocity: 42 SP
   • 3 offene Stories aufgrund von API-Problemen
3. Besprechung der aktuellen Herausforderungen:
   • Detaillierte Diskussion zu den API-Problemen und deren Auswirkungen auf den Sprint.
4. Deployment-Plan:
   • Deployment ist für Freitag geplant.
5. Offene Action Items aus dem letzten Meeting:
   • Ben: API-Fix bis heute liefern.
   • Clara: Erstellen des Test-Reports bis Donnerstag.
6. Abschlussrunde:
   • Offene Fragen, Feedback und nächste Schritte.

Dieses Briefing bietet einen klaren Überblick und leitet die Agenda des Meetings strukturiert ein.

# 8 | Demo 3 — Mehrdeutige Anfrage mit Skill-Auswahl

---

**Erwartetes Verhalten:** Die Anfrage berührt mehrere Skill-Bereiche.
Der Agent muss entscheiden: Recherche-Skill? Compliance-Skill? Kombination?

Dieses Demo zeigt die **Grenzen und Stärken** des automatischen Routings:
- Bei klaren Trigger-Phrasen → eindeutiger Match
- Bei Überschneidungen → Agent priorisiert oder kombiniert
- Bei keinem Match → direkter Antwort ohne Skill

In [39]:
# Demo 3: Mehrdeutige Anfrage — Recherche + Compliance-Bezug

anfrage_mixed = """
Ich bereite ein Meeting mit unserem neuen Forschungspartner aus Südkorea vor.
Das Thema ist KI-gestützte Qualitätsprüfung in der Fertigung.
Kannst du mir relevante Hintergrundinformationen zum Thema zusammenstellen
und gleichzeitig checken, ob es besondere Compliance-Anforderungen für
Technologie-Kooperationen mit koreanischen Firmen gibt?
"""

print("📋 Anfrage:")
print(anfrage_mixed.strip())
print("\n" + "─" * 60 + "\n")

antwort_3 = invoke_agent(anfrage_mixed, session_id="demo3-mixed")

print("🤖 Agent-Antwort:")
mprint(antwort_3)

📋 Anfrage:
Ich bereite ein Meeting mit unserem neuen Forschungspartner aus Südkorea vor.
Das Thema ist KI-gestützte Qualitätsprüfung in der Fertigung.
Kannst du mir relevante Hintergrundinformationen zum Thema zusammenstellen
und gleichzeitig checken, ob es besondere Compliance-Anforderungen für
Technologie-Kooperationen mit koreanischen Firmen gibt?

────────────────────────────────────────────────────────────

🤖 Agent-Antwort:


Hier sind die gesammelten Hintergrundinformationen:

1. KI-gestützte Qualitätsprüfung in der Fertigung:
   • Moderne Systeme nutzen Computer Vision und Machine Learning, um Produktionsfehler frühzeitig und automatisiert zu erkennen.
   • Durch den Einsatz von bildbasierten Inspektionssystemen können auch kleinste Defekte in Echtzeit identifiziert werden, was die Produktqualität steigert.
   • Vorhersagemodelle ermöglichen die Prognose von Maschinenausfällen und leisten einen Beitrag zur vorausschauenden Wartung.
   • Die Integration solcher Systeme führt oft zu einer Reduzierung von Ausschuss und Prozesskosten, da Fehler schneller behoben werden und Produktionsprozesse optimiert werden.
   • Studien und Praxisbeispiele zeigen, dass KI-basierte Qualitätsprüfungen besonders in Bereichen mit hoher Automatisierung und großen Produktionsvolumina signifikante Verbesserungen bringen.

2. Compliance-Anforderungen bei Technologie-Kooperationen mit koreanischen Firmen:
   • Der durchgeführte Compliance-Check ergab ein geringes Risiko („low“) für Forschungspartner aus Südkorea.
   • Dennoch sollten allgemeine Datenschutzbestimmungen, Exportkontrollvorgaben und IP-Schutzregelungen beachtet werden.
   • Es kann sinnvoll sein, spezifische vertragliche Regelungen zur Offenlegung, Datenverarbeitung und Kooperation im Technologietransfer zu definieren, um unerwartete rechtliche Herausforderungen zu vermeiden.
   • Regelmäßige Überprüfungen der gesetzlichen Änderungen sowohl im Inland als auch in Korea helfen, jederzeit compliant zu bleiben.

Diese Informationen bieten einen fundierten Rahmen für dein Meeting und beleuchten sowohl die technischen Chancen der KI-gestützten Qualitätsprüfung als auch die zu beachtenden Compliance-Aspekte.

In [40]:
# 8.1 Analyse: Welcher Skill wurde gewählt?
#
# Zeigt den Mechanismus: Welche Frontmatter-descriptions hat der Agent gesehen?
# Und wie hat er entschieden?

zeilen = [
    "## 🔍 Skill-Routing-Analyse für Demo 3",
    "",
    "| Skill | Trigger-Signal in Anfrage | Erwarteter Match |",
    "|-------|--------------------------|------------------|",
    "| compliance | 'Compliance-Anforderungen', 'koreanischen Firmen' | ⚠️ Teilmatch |",
    "| meeting-briefing | 'Meeting vorbereiten', 'Hintergrund' | ⚠️ Teilmatch |",
    "| research | 'Hintergrundinformationen', 'zusammenstellen' | ✅ Starker Match |",
    "",
    "> 💡 Bei Überschneidungen wählt der Agent den stärksten Match",
    "> oder kombiniert Skills — je nach LLM-Entscheidung.",
    "> Das Verhalten ist nicht deterministisch und kann variieren.",
]
mprint("\n".join(zeilen))

## 🔍 Skill-Routing-Analyse für Demo 3

| Skill | Trigger-Signal in Anfrage | Erwarteter Match |
|-------|--------------------------|------------------|
| compliance | 'Compliance-Anforderungen', 'koreanischen Firmen' | ⚠️ Teilmatch |
| meeting-briefing | 'Meeting vorbereiten', 'Hintergrund' | ⚠️ Teilmatch |
| research | 'Hintergrundinformationen', 'zusammenstellen' | ✅ Starker Match |

> 💡 Bei Überschneidungen wählt der Agent den stärksten Match
> oder kombiniert Skills — je nach LLM-Entscheidung.
> Das Verhalten ist nicht deterministisch und kann variieren.

# 9 | Vergleich: natives Skill-System vs. M33-Ansatz

---

Beide Ansätze sind didaktisch wertvoll und ergänzen sich:

| Dimension | M33 (manuell, Gate-Writer) | M36 (nativ, Progressive Disclosure) |
|-----------|--------------------------|--------------------------------------|
| **Skill-Loading** | Explizit via `urllib.request` | Automatisch via `skills=[...]` |
| **Skill-Quelle** | GitHub Raw URL (direkt) | GitHub → lokaler Cache |
| **Routing** | Fest verdrahtet (ein Skill) | Dynamisch durch LLM |
| **Anzahl Skills** | 1 (meeting-briefing) | 3 (compliance, meeting, research) |
| **Gate-Writer** | ✅ Vollständig (o3 → gpt-5.1) | ❌ Nicht implementiert |
| **Kontrolle** | ✅ Sehr hoch | ⚠️ Mittel (LLM entscheidet) |
| **Transparenz** | ✅ Jeder Schritt sichtbar | ⚠️ Routing ist black-box |
| **Erweiterbarkeit** | Aufwendig (neuer Code) | ✅ Neuer Ordner + SKILL.md |
| **Reproduzierbarkeit** | ✅ Hoch (SHA-pinned) | ✅ Hoch (SHA-pinned + Version) |

**Wann welchen Ansatz wählen?**

```
Genau ein Skill, kritische Logik, Gate-Writer-Qualität   → M33-Ansatz
Mehrere Skills, flexible Erweiterung, Rapid Prototyping  → M36-Ansatz
Produktion mit Audit-Trail                               → M33-Ansatz
Interne Tools / Wissensdatenbank                         → M36-Ansatz
```

In [41]:
#@markdown   <p><font size="4" color="green">Entscheidungsbaum: M33 vs. M36</font></p>

diagram_compare = '''
%%{init: {'theme':'light'}}%%
flowchart TB
    START([" ❓ Neues Skill-System bauen"]) --> Q1{Wie viele Skills?}

    Q1 -->|"genau 1"| Q2{Gate-Writer
Qualität nötig?}
    Q1 -->|"> 1"| Q3{Routing flexibel
bzw. erweiterbar?}

    Q2 -->|"ja"| M33[" 🔵 M33-Ansatz\nManuelle Kontrolle"]
    Q2 -->|"nein"| M36A[" 🟢 M36-Ansatz\nEinfacher"]

    Q3 -->|"ja"| M36B[" 🟢 M36-Ansatz\nProgressive Disclosure"]
    Q3 -->|"nein"| Q4{Audit-Trail
erforderlich?}

    Q4 -->|"ja"| M33
    Q4 -->|"nein"| M36B
'''
mermaid(diagram_compare, width=550)

# 10 | Grenzen, Stabilität, didaktische Einordnung

---

## Bekannte Grenzen

| Grenze | Details |
|--------|---------|
| **Lokales FS erforderlich** | `skills=[...]` akzeptiert keine GitHub-URLs direkt — Cache-Schritt ist immer nötig |
| **Nicht deterministisch** | LLM-Routing kann bei ähnlichen Trigger-Phrasen variieren |
| **Kein Gate-Writer** | Natives Skill-System hat keine eingebaute zweistufige Analyse |
| **Versions-API** | `skills=`-Parameter ist in v0.4 noch nicht als stable markiert |
| **Debugging** | Schwieriger als M33: Welcher Skill gewählt wurde, ist nicht direkt sichtbar |

## Stabilität (Stand: März 2026)

```python
# Getestete, stabile Kombination
deepagents==0.4.12
langchain>=1.0.0
langgraph>=1.0.0
```

> ⚠️ **Vor einem Versions-Update:** `/check-deepagents`-Command ausführen
> und das M36-CHANGELOG überprüfen.

## Didaktische Einordnung im Kurs

```
M31  Agent-Skill-Compliance     → Skill-Konzept einführen
M32  DeepAgents-Harness         → Harness-Grundlagen
M33  Meeting-Briefing-Skill     → Skill-Tiefe: Gate-Writer, explizite Kontrolle
M36  Multi-Skill Progressive    → Skill-Breite: Routing, Erweiterbarkeit
```

M33 und M36 sind **keine Alternativen**, sondern **komplementäre Lernpfade**.
Beide zusammen ergeben das vollständige Bild moderner Skill-basierter Agenten.